##### Import the libraries


In [156]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import holidays
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score)
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import joblib
import os
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

##### Load the prediction files

In [157]:
sarima = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Baseline Models\Predictions\sarima_predictions.csv")
holt = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Baseline Models\Predictions\holt_predictions.csv")
xgb = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Advanced Models\Predictions\xgboost_predictions.csv")
sarimax = pd.read_csv(r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Advanced Models\Predictions\sarimax_predictions.csv")

##### Add the model name

In [158]:
sarima["model"] = "SARIMA"
holt["model"] = "Holt-Winters"
xgb["model"] = "XGBoost"
sarimax["model"] = "SARIMAX"

##### Combine everything

In [159]:
all_predictions = pd.concat([sarima, holt, xgb, sarimax], ignore_index=True)
all_predictions.head()

,Unit,Date,Actual,Predicted,model
0,HHN-BIR-01_ICU,2025-12-25,9.000000,8.539481,SARIMA
1,HHN-BIR-01_ICU,2025-12-26,9.000000,8.032897,SARIMA
2,HHN-BIR-01_ICU,2025-12-27,8.500000,7.752455,SARIMA
3,HHN-BIR-01_ICU,2025-12-28,7.541667,7.709320,SARIMA
4,HHN-BIR-01_ICU,2025-12-29,8.791667,7.668268,SARIMA


##### Rename columns to standard names

In [160]:
all_predictions.columns = (all_predictions.columns.str.lower().str.strip())

In [161]:
all_predictions.columns

Index(['unit', 'date', 'actual', 'predicted', 'model'], dtype='object')

##### Calculate MAE, RMSE, MAPE

In [162]:
def calculate_metrics(group):

    actual = group["actual"]
    predicted = group["predicted"]

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    # avoid division by zero
    mape = np.mean(
        np.abs(
            (actual - predicted) / actual.replace(0, np.nan)
        )
    ) * 100

    return pd.Series({
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    })

##### Evaluate every model for every unit

In [163]:
evaluation_results = (
    all_predictions
    .groupby(
        ["unit", "model"]
    )
    .apply(calculate_metrics)
    .reset_index()
)


evaluation_results

,unit,model,MAE,RMSE,MAPE
0,HHN-BIR-01_Cardiology Ward,Holt-Winters,2.635654,3.556100,16.579061
1,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359
2,HHN-BIR-01_Cardiology Ward,SARIMAX,3.948593,5.828995,19.793388
3,HHN-BIR-01_Cardiology Ward,XGBoost,2.691129,3.762878,12.217064
4,HHN-BIR-01_Day Case Unit,Holt-Winters,0.298985,0.324769,115.708812
...,...,...,...,...,...
155,HHN-MAN-01_Orthopaedics Ward A,XGBoost,2.948937,3.543631,13.932180
156,HHN-MAN-01_Orthopaedics Ward B,Holt-Winters,1.132514,1.628536,5.728951
157,HHN-MAN-01_Orthopaedics Ward B,SARIMA,1.207932,1.565897,6.189143
158,HHN-MAN-01_Orthopaedics Ward B,SARIMAX,4.448138,5.422910,17.969923


##### Select the best model per unit

In [164]:
best_models = (
    evaluation_results
    .sort_values(
        by=["unit", "RMSE"]
    )
    .groupby("unit")
    .first()
    .reset_index()
)


best_models

,unit,model,MAE,RMSE,MAPE
0,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359
1,HHN-BIR-01_Day Case Unit,XGBoost,0.160609,0.208616,71.544745
2,HHN-BIR-01_General Medicine Ward A,Holt-Winters,0.216116,0.250543,1.365993
3,HHN-BIR-01_General Medicine Ward B,XGBoost,1.161449,1.514744,8.634891
4,HHN-BIR-01_ICU,Holt-Winters,0.284338,0.487589,3.549822
5,HHN-BIR-01_Oncology Ward,Holt-Winters,1.734271,1.845498,16.813870
6,HHN-BIR-01_Orthopaedics Ward A,Holt-Winters,0.085957,0.126234,0.789861
7,HHN-BIR-01_Orthopaedics Ward B,XGBoost,0.174042,0.283800,1.689507
8,HHN-EDI-01_Cardiology Ward,Holt-Winters,1.628670,1.805807,11.849799
9,HHN-EDI-01_Day Case Unit,XGBoost,0.157696,0.192113,76.691431


In [165]:
best_models = (
    evaluation_results
    .sort_values(by=["unit", "RMSE"])
    .groupby("unit")
    .first()
    .reset_index()
)

best_models

,unit,model,MAE,RMSE,MAPE
0,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359
1,HHN-BIR-01_Day Case Unit,XGBoost,0.160609,0.208616,71.544745
2,HHN-BIR-01_General Medicine Ward A,Holt-Winters,0.216116,0.250543,1.365993
3,HHN-BIR-01_General Medicine Ward B,XGBoost,1.161449,1.514744,8.634891
4,HHN-BIR-01_ICU,Holt-Winters,0.284338,0.487589,3.549822
5,HHN-BIR-01_Oncology Ward,Holt-Winters,1.734271,1.845498,16.813870
6,HHN-BIR-01_Orthopaedics Ward A,Holt-Winters,0.085957,0.126234,0.789861
7,HHN-BIR-01_Orthopaedics Ward B,XGBoost,0.174042,0.283800,1.689507
8,HHN-EDI-01_Cardiology Ward,Holt-Winters,1.628670,1.805807,11.849799
9,HHN-EDI-01_Day Case Unit,XGBoost,0.157696,0.192113,76.691431


In [166]:
best_models[["hospital_id", "ward"]] = (
    best_models["unit"]
    .str.split("_", n=1, expand=True)
)                                                                                          
best_models.head()

,unit,model,MAE,RMSE,MAPE,hospital_id,ward
0,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359,HHN-BIR-01,Cardiology Ward
1,HHN-BIR-01_Day Case Unit,XGBoost,0.160609,0.208616,71.544745,HHN-BIR-01,Day Case Unit
2,HHN-BIR-01_General Medicine Ward A,Holt-Winters,0.216116,0.250543,1.365993,HHN-BIR-01,General Medicine Ward A
3,HHN-BIR-01_General Medicine Ward B,XGBoost,1.161449,1.514744,8.634891,HHN-BIR-01,General Medicine Ward B
4,HHN-BIR-01_ICU,Holt-Winters,0.284338,0.487589,3.549822,HHN-BIR-01,ICU


In [167]:
best_models = best_models[
    [
        "hospital_id",
        "unit",
        "model",
        "MAE",
        "RMSE",
        "MAPE"
    ]
]


best_models

,hospital_id,unit,model,MAE,RMSE,MAPE
0,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359
1,HHN-BIR-01,HHN-BIR-01_Day Case Unit,XGBoost,0.160609,0.208616,71.544745
2,HHN-BIR-01,HHN-BIR-01_General Medicine Ward A,Holt-Winters,0.216116,0.250543,1.365993
3,HHN-BIR-01,HHN-BIR-01_General Medicine Ward B,XGBoost,1.161449,1.514744,8.634891
4,HHN-BIR-01,HHN-BIR-01_ICU,Holt-Winters,0.284338,0.487589,3.549822
5,HHN-BIR-01,HHN-BIR-01_Oncology Ward,Holt-Winters,1.734271,1.845498,16.813870
6,HHN-BIR-01,HHN-BIR-01_Orthopaedics Ward A,Holt-Winters,0.085957,0.126234,0.789861
7,HHN-BIR-01,HHN-BIR-01_Orthopaedics Ward B,XGBoost,0.174042,0.283800,1.689507
8,HHN-EDI-01,HHN-EDI-01_Cardiology Ward,Holt-Winters,1.628670,1.805807,11.849799
9,HHN-EDI-01,HHN-EDI-01_Day Case Unit,XGBoost,0.157696,0.192113,76.691431


In [168]:
hospital_mapping = pd.DataFrame({
    "hospital_id": [
        "HHN-BIR-01",
        "HHN-EDI-01",
        "HHN-LON-01",
        "HHN-LON-02",
        "HHN-MAN-01"
    ],
    "hospital_name": [
        "Horizon Birmingham",
        "Horizon Edinburgh",
        "Horizon London Central",
        "Horizon London Riverside",
        "Horizon Manchester"
    ]
})

In [169]:
best_models = best_models.merge(
    hospital_mapping,
    on="hospital_id",
    how="left"
)
best_models.head()

,hospital_id,unit,model,MAE,RMSE,MAPE,hospital_name
0,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359,Horizon Birmingham
1,HHN-BIR-01,HHN-BIR-01_Day Case Unit,XGBoost,0.160609,0.208616,71.544745,Horizon Birmingham
2,HHN-BIR-01,HHN-BIR-01_General Medicine Ward A,Holt-Winters,0.216116,0.250543,1.365993,Horizon Birmingham
3,HHN-BIR-01,HHN-BIR-01_General Medicine Ward B,XGBoost,1.161449,1.514744,8.634891,Horizon Birmingham
4,HHN-BIR-01,HHN-BIR-01_ICU,Holt-Winters,0.284338,0.487589,3.549822,Horizon Birmingham


In [170]:
best_models = best_models[
    [
        "hospital_name",
        "hospital_id",
        "unit",
        "model",
        "MAE",
        "RMSE",
        "MAPE"
    ]
]

##### Save your final results

In [171]:
best_models.to_csv("../Models/Best Models/best_models.csv", index=False)

In [174]:
best_models.head()

,hospital_name,hospital_id,unit,model,MAE,RMSE,MAPE
0,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,SARIMA,2.149314,2.866235,13.236359
1,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Day Case Unit,XGBoost,0.160609,0.208616,71.544745
2,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward A,Holt-Winters,0.216116,0.250543,1.365993
3,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_General Medicine Ward B,XGBoost,1.161449,1.514744,8.634891
4,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_ICU,Holt-Winters,0.284338,0.487589,3.549822


Copy only winning trained models

In [ ]:
import os
import shutil


# Baseline models location
baseline_path = r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Baseline Models\Trained Models"

# Advanced models location
advanced_path = r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Advanced Models\Trained Models"

# Final destination
best_models_path = r"C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Hospital_Bed_Demand_Forecasting\Models\Beat Models\Best Trained Models"


model_info = {

    "Holt-Winters": {
        "base": baseline_path,
        "folder": "Holt",
        "prefix": "holt_"
    },

    "SARIMA": {
        "base": baseline_path,
        "folder": "SARIMA",
        "prefix": "sarima_"
    },

    "SARIMAX": {
        "base": baseline_path,
        "folder": "SARIMAX",
        "prefix": "sarimax_"
    },

    "XGBoost": {
        "base": advanced_path,
        "folder": "XGBoost",
        "prefix": "xgboost_"
    }
}


for _, row in best_models.iterrows():

    unit = row["unit"]
    model_name = row["model"]


    model_details = model_info[model_name]


    filename = (
        f"{model_details['prefix']}{unit}.pkl"
    )


    source_path = os.path.join(
        model_details["base"],
        model_details["folder"],
        filename
    )


    destination_path = os.path.join(
        best_models_path,
        filename
    )


    if os.path.exists(source_path):

        shutil.copy(
            source_path,
            destination_path
        )

        print("Saved:", filename)

    else:

        print("Missing:", source_path)

Saved: sarima_HHN-BIR-01_Cardiology Ward.pkl
Saved: xgboost_HHN-BIR-01_Day Case Unit.pkl
Saved: holt_HHN-BIR-01_General Medicine Ward A.pkl
Saved: xgboost_HHN-BIR-01_General Medicine Ward B.pkl
Saved: holt_HHN-BIR-01_ICU.pkl
Saved: holt_HHN-BIR-01_Oncology Ward.pkl
Saved: holt_HHN-BIR-01_Orthopaedics Ward A.pkl
Saved: xgboost_HHN-BIR-01_Orthopaedics Ward B.pkl
Saved: holt_HHN-EDI-01_Cardiology Ward.pkl
Saved: xgboost_HHN-EDI-01_Day Case Unit.pkl
Saved: holt_HHN-EDI-01_General Medicine Ward A.pkl
Saved: holt_HHN-EDI-01_General Medicine Ward B.pkl
Saved: holt_HHN-EDI-01_ICU.pkl
Saved: holt_HHN-EDI-01_Oncology Ward.pkl
Saved: holt_HHN-EDI-01_Orthopaedics Ward A.pkl
Saved: holt_HHN-EDI-01_Orthopaedics Ward B.pkl
Saved: sarimax_HHN-LON-01_Cardiology Ward.pkl
Saved: sarima_HHN-LON-01_Day Case Unit.pkl
Saved: xgboost_HHN-LON-01_General Medicine Ward A.pkl
Saved: sarima_HHN-LON-01_General Medicine Ward B.pkl
Saved: holt_HHN-LON-01_ICU.pkl
Saved: xgboost_HHN-LON-01_Oncology Ward.pkl
Saved: xgbo

##### Merge Predictions With Best Models

In [176]:
best_predictions = pd.merge(all_predictions, best_models, on=["unit", "model"], how="inner")
best_predictions.head()

,unit,date,actual,predicted,model,hospital_name,hospital_id,MAE,RMSE,MAPE
0,HHN-BIR-01_Cardiology Ward,2025-12-25,21.208333,22.281654,SARIMA,Horizon Birmingham,HHN-BIR-01,2.149314,2.866235,13.236359
1,HHN-BIR-01_Cardiology Ward,2025-12-26,23.375000,21.692058,SARIMA,Horizon Birmingham,HHN-BIR-01,2.149314,2.866235,13.236359
2,HHN-BIR-01_Cardiology Ward,2025-12-27,19.000000,19.730131,SARIMA,Horizon Birmingham,HHN-BIR-01,2.149314,2.866235,13.236359
3,HHN-BIR-01_Cardiology Ward,2025-12-28,18.875000,17.138078,SARIMA,Horizon Birmingham,HHN-BIR-01,2.149314,2.866235,13.236359
4,HHN-BIR-01_Cardiology Ward,2025-12-29,16.916667,16.988289,SARIMA,Horizon Birmingham,HHN-BIR-01,2.149314,2.866235,13.236359


##### Reorder columns

In [179]:
best_predictions = best_predictions[[
        "hospital_name",
        "hospital_id",
        "unit",
        "date",
        "actual",
        "predicted",
        "model"
    ]]
print(best_predictions.shape)

best_predictions.head()

(648, 7)


,hospital_name,hospital_id,unit,date,actual,predicted,model
0,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-25,21.208333,22.281654,SARIMA
1,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-26,23.375000,21.692058,SARIMA
2,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-27,19.000000,19.730131,SARIMA
3,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-28,18.875000,17.138078,SARIMA
4,Horizon Birmingham,HHN-BIR-01,HHN-BIR-01_Cardiology Ward,2025-12-29,16.916667,16.988289,SARIMA


In [180]:
print(
    "Number of Units:",
    best_predictions["unit"].nunique()
)

Number of Units: 40


##### Save best models predictions

In [181]:
best_models.to_csv("../Models/Best Models/Best Models Predictions/best_models_predictions.csv", index=False)